<a href="https://colab.research.google.com/github/yuhan222/114-2-Programing-Language/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_Part_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一 part2）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1lLcOA_CUkl6N0H6OiAUnQyBIUyRR45Q3CHSrPoK7vCE/edit?usp=sharing

In [4]:
# 1. 安裝 Gradio 套件 (Colab 預設沒裝，必須先執行這行)
!pip install gradio -q
# 下載中文字體 (台北黑體)
!wget -O font.ttf https://github.com/notofonts/noto-cjk/raw/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf

# 2. 匯入必要套件
import gspread
import pandas as pd
import gradio as gr
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from google.colab import auth
from google.auth import default
from datetime import datetime

# 註冊字體
fm.fontManager.addfont('font.ttf')
prop = fm.FontProperties(fname='font.ttf')
# 設定為預設字體
plt.rcParams['font.family'] = prop.get_name()

# 3. Google 帳號授權
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 4. 定義你的試算表 URL (建議將網址抽出來變成變數，方便之後維護)
SHEET_URL = 'https://docs.google.com/spreadsheets/d/1lLcOA_CUkl6N0H6OiAUnQyBIUyRR45Q3CHSrPoK7vCE/edit?usp=sharing'

# 5. 開啟試算表物件 (設為全域變數，後續 Gradio 函數才抓得到)
try:
    gsheets = gc.open_by_url(SHEET_URL)
    # 預先確認「記帳本」分頁存在
    worksheet = gsheets.worksheet('記帳本')
    print("✅ 成功連線至 Google Sheet：", gsheets.title)
except Exception as e:
    print("❌ 連線失敗，請檢查網址或權限：", e)

--2026-03-13 08:05:54--  https://github.com/notofonts/noto-cjk/raw/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/notofonts/noto-cjk/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf [following]
--2026-03-13 08:05:55--  https://raw.githubusercontent.com/notofonts/noto-cjk/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16435884 (16M) [application/octet-stream]
Saving to: ‘font.ttf’

font.ttf            100%[===================>]  15.67M  --.-KB/s    in 0.1s    

2026-03

In [5]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt

def handle_accounting_logic(date_str, time_str, sort, other_sort, item, amount, person, other_person, place, methods, remark, budget):
    try:
        # 1. 基本設定與重複檢查 (保留原邏輯)
        final_sort = other_sort if sort == "其他" else sort
        final_payer = other_person if person == "其他" else person
        worksheet = gsheets.worksheet('記帳本')

        # 2. 寫入資料 (省略重複檢查代碼，保持簡潔)
        row_to_write = [date_str, time_str, final_sort, item, amount, final_payer, place, methods, remark]
        worksheet.append_row(values=row_to_write, value_input_option='USER_ENTERED')

        # 3. 讀取與計算
        all_records = worksheet.get_all_values()
        df = pd.DataFrame(all_records[1:], columns=all_records[0])
        df['金額'] = pd.to_numeric(df['金額'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

        category_series = df.groupby('分類')['金額'].sum()
        total_spent = category_series.sum()

        # --- ✨ 新增：預算進度條邏輯 ---
        budget_val = float(budget) if budget else 20000
        usage_rate = (total_spent / budget_val) * 100
        remaining = budget_val - total_spent

        # 製作簡易進度條 [■■■□□]
        bar_length = 20
        filled_length = int(bar_length * min(usage_rate / 100, 1))
        bar = "■" * filled_length + "□" * (bar_length - filled_length)

        # 4. 組合摘要文字
        summary_report = f"📊 【本月支出總計】：${total_spent:,.0f} / ${budget_val:,.0f}\n"
        summary_report += f"📈 預算進度：[{bar}] {usage_rate:.1f}%\n"
        summary_report += f"💰 剩餘額度：${remaining:,.0f}\n"
        summary_report += "----------------------------\n"
        summary_report += "🗂 【分類明細】：\n"
        for cat, val in category_series.items():
            summary_report += f" • {cat}: ${val:,.0f}\n"

        summary_report += "----------------------------\n"

        # --- ✨ 教練語錄升級 ---
        summary_report += "🌵 【毒舌教練語錄】：\n"
        if usage_rate >= 100:
            ai_comment = "預算已經爆了！你現在是打算破產還是去睡公園？別再點滑鼠了！"
        elif usage_rate >= 80:
            ai_comment = f"預算只剩下 {usage_rate:.1f}%，你的錢包正在加護病房插管，你還在買？"
        elif remaining < 1000:
            ai_comment = "剩不到一千塊，你下半個月是打算靠光合作用過活嗎？"
        else:
            ai_comment = "雖然還沒爆，但看你這花錢的速度，我看也快了。"

        summary_report += f"「{ai_comment}」"

        # 5. 繪圖 (與之前相同)
        fig, ax = plt.subplots(figsize=(6, 5))
        category_series.plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=140)
        ax.set_title("Expenditure Distribution")

        return "✅ 寫入成功！", summary_report, fig

    except Exception as e:
        return f"❌ 錯誤", str(e), plt.figure()

In [6]:
import gradio as gr
from datetime import datetime
import pandas as pd

# 定義抓取歷史紀錄的函數
def get_history_data():
    try:
        worksheet = gsheets.worksheet('記帳本')
        all_records = worksheet.get_all_values()
        if len(all_records) > 1:
            # 轉換為 DataFrame，排除標題列
            df = pd.DataFrame(all_records[1:], columns=all_records[0])
            return df
        else:
            return pd.DataFrame(columns=["尚未有資料"])
    except Exception as e:
        return pd.DataFrame(columns=[f"讀取失敗: {e}"])

with gr.Blocks(theme=gr.themes.Soft(), title="🏠 雲端支出統計系統") as demo:
    gr.Markdown("# 💸 智能雲端記帳管理系統")

    # --- ✨ 使用 Tabs 建立分頁 ---
    with gr.Tabs():

        # 分頁一：原本的記帳功能
        with gr.TabItem("📝 記帳與分析"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### ⚙️ 基礎設定")
                    budget_in = gr.Number(label="💰 本月總預算設定 (NTD)", value=15000, precision=0)
                    gr.Markdown("---")
                    gr.Markdown("### 📥 消費資料輸入")
                    with gr.Row():
                        date_in = gr.Textbox(label="日期", value=datetime.now().strftime('%Y-%m-%d'), lines=1)
                        time_in = gr.Textbox(label="時間", value=datetime.now().strftime('%H:%M'), lines=1)

                    sort_in = gr.Dropdown(choices=["外食", "交通", "日常用品", "娛樂", "醫藥", "其他"], label="分類", value="外食")
                    other_sort_in = gr.Textbox(label="自定義分類", visible=False, lines=1)
                    sort_in.change(lambda x: gr.update(visible=(x=="其他")), sort_in, other_sort_in)

                    item_in = gr.Textbox(label="品項", placeholder="例如：咖啡", lines=1)
                    amount_in = gr.Textbox(label="金額 (NTD)", value="0", lines=1)

                    person_in = gr.Radio(choices=["我", "其他"], label="付款人", value="我")
                    other_person_in = gr.Textbox(label="請輸入付款人姓名", visible=False, placeholder="例如：王小明", lines=1)
                    person_in.change(lambda x: gr.update(visible=(x=="其他")), person_in, other_person_in)

                    with gr.Accordion("更多詳細資訊 (選填)", open=False):
                        place_in = gr.Textbox(label="地點", placeholder="例如：便利商店", lines=1)
                        method_in = gr.Dropdown(choices=["現金", "信用卡", "LinePay", "ApplePay"], label="支付方式", value="現金")
                        remark_in = gr.Textbox(label="備註", placeholder="無", lines=1)

                    with gr.Row():
                        submit_btn = gr.Button("🚀 點我送出並統計", variant="primary")
                        reset_btn = gr.Button("🧹 清空消費資料")

                with gr.Column():
                    gr.Markdown("### 📈 消費統計摘要")
                    status_out = gr.Textbox(label="執行狀態", interactive=False, lines=1)
                    analysis_out = gr.Textbox(label="支出累計與分類分析", lines=15, interactive=False)
                    plot_out = gr.Plot(label="消費佔比分佈圖")

        # 分頁二：檢視歷史資料
        with gr.TabItem("📜 歷史紀錄檢視"):
            gr.Markdown("### 🔍 原始資料表")
            refresh_btn = gr.Button("🔄 重新整理歷史紀錄", variant="secondary")

            # 使用 Dataframe 元件來展示資料
            history_table = gr.Dataframe(
                headers=["日期", "時間", "分類", "品項", "金額", "付款人", "地點", "支付方式", "備註"],
                datatype=["str", "str", "str", "str", "number", "str", "str", "str", "str"],
                interactive=False, # 僅查看，不編輯
                wrap=True
            )

    # --- 功能串接 ---

    # 1. 提交邏輯
    submit_btn.click(
        fn=handle_accounting_logic,
        inputs=[date_in, time_in, sort_in, other_sort_in, item_in, amount_in, person_in, other_person_in, place_in, method_in, remark_in, budget_in],
        outputs=[status_out, analysis_out, plot_out]
    )

    # 2. 重置邏輯 (保留預算設定)
    def reset_consumption_only():
        return (datetime.now().strftime('%Y-%m-%d'), datetime.now().strftime('%H:%M'), "外食", "", "", "0", "我", "", "", "現金", "無")

    reset_btn.click(
        fn=reset_consumption_only,
        inputs=[],
        outputs=[date_in, time_in, sort_in, other_sort_in, item_in, amount_in, person_in, other_person_in, place_in, method_in, remark_in]
    )

    # 3. 刷新歷史紀錄表格
    refresh_btn.click(
        fn=get_history_data,
        inputs=[],
        outputs=history_table
    )

demo.launch(share=True, debug=True)

/tmp/ipykernel_1326/480079977.py:19: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="🏠 雲端支出統計系統") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ae18ca60f42a79411c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://ae18ca60f42a79411c.gradio.live
